In [4]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA

/Users/shorya/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/shorya/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
MODEL_NAME = "llama3-8b-8192"

if GROQ_API_KEY is None:
    raise ValueError("GROQ_API_KEY not found")

In [6]:
from langchain_community.document_loaders import PyMuPDFLoader
DATA_PATH = [
    "/Users/shorya/Desktop/Lead-to-Deal/.vscode/chatbot/Data/Blockchain-Book.pdf",
    "/Users/shorya/Desktop/Lead-to-Deal/.vscode/chatbot/Data/CC-Book.pdf",
    "/Users/shorya/Desktop/Lead-to-Deal/.vscode/chatbot/Data/CG-Book.pdf",
    "/Users/shorya/Desktop/Lead-to-Deal/.vscode/chatbot/Data/CN-Book.pdf",
    "/Users/shorya/Desktop/Lead-to-Deal/.vscode/chatbot/Data/DSA-Book.pdf"
    
]

documents = []

for pdf in DATA_PATH:
    loader = PyMuPDFLoader(pdf)
    docs = loader.load()
    documents.extend(docs)

print("Total documents loaded:", len(documents))

Total documents loaded: 1966


In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))

Total chunks created: 6814


In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded successfully.")

/var/folders/4g/tsqhj6z10kscb7b6nft8pcd80000gn/T/ipykernel_7966/1283672961.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Embeddings model loaded successfully.


In [6]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

# Save locally
vectorstore.save_local("faiss_index")

print("Vectorstore created and saved successfully.")

Vectorstore created and saved successfully.


In [11]:
vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print("Vectorstore loaded successfully.")

Vectorstore loaded successfully.


In [12]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [13]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama3-8b-8192"
)

print("LLM loaded successfully.")

LLM loaded successfully.


In [10]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

print("RAG chain created successfully.")

RAG chain created successfully.


In [14]:
print("🎓 CSE Chatbot Ready! Type 'exit' to stop.\n")

while True:
    user_question = input("Ask your question: ")

    if user_question.lower() == "exit":
        break

    query = f"""
    You are a Computer Science professor.

    Student Level: 2nd Year B.Tech CSE
    Answer in exam-oriented structured format:
    1. Definition
    2. Explanation
    3. Example
    4. Key Points
    5. Conclusion

    Question:
    {user_question}
    """

    response = qa_chain({"query": query})

    print("\n📘 Answer:\n")
    print(response["result"])

    print("\n📚 Sources Used:")
    for doc in response["source_documents"]:
        print(doc.metadata["source"])

    print("\n" + "-"*60 + "\n")

🎓 CSE Chatbot Ready! Type 'exit' to stop.



NameError: name 'qa_chain' is not defined